In [1]:
from helpers.models import Models
from helpers.llm_client import LLMClient
from helpers.functions import *
import pandas as pd
import os
from sklearn.metrics import accuracy_score

pd.set_option('display.max_rows', None)    # Show all rows
# pd.set_option('display.max_colwidth', None)  # Show full column width

/home/jimbo/Desktop/GSoC24/repo/GSoC24/gsoc24env/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/home/jimbo/Desktop/GSoC24/repo/GSoC24/gsoc24env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
df = pd.read_csv('results/test-1.csv')
len(df)

200

In [2]:
# Check to make sure that all API keys are present
os.environ['GROQ_API_KEY'] 
os.environ['NVIDIA_API_KEY']
os.environ['TOGETHER_API_KEY']    
'OK'
#

'OK'

In [3]:
test_file_path = 'extras/LastGoodNomosTestfilesScan.txt'
test_file_columns = ['file path', 'licenses']

with open(test_file_path, 'r') as file:
    test_data = file.readlines()

test_df = pd.DataFrame(columns=test_file_columns)

for line in test_data:
    if "contains license(s)" in line:  # Check for the expected pattern
        licenses = line.split("contains license(s) ")[1].strip().split("\\")[0]
        file_path = line.split('File ')[1].split(' contains license(s)')[0].strip()
        licenses = str(licenses)
        licenses = licenses.split(',')
        licenses = '\n'.join(licenses)
        temp_df = pd.DataFrame({'file path': file_path, 'licenses': [licenses]})
        test_df = pd.concat([test_df, temp_df], ignore_index=True)

test_df.loc[1997, 'licenses']

'Dual-license\nGPL\nToolbar2000'

In [4]:
for index, row in test_df.iterrows():
    try:
        with open(os.path.join('extras', row['file path']), "r", encoding='utf-8') as f:
            comments = f.read()
    except:
        print(f'Dropping Index: {index}')
        test_df = test_df.drop(index)
test_df = test_df.reset_index()

Dropping Index: 39
Dropping Index: 83
Dropping Index: 160
Dropping Index: 161
Dropping Index: 162
Dropping Index: 202
Dropping Index: 301
Dropping Index: 322
Dropping Index: 350
Dropping Index: 548
Dropping Index: 625
Dropping Index: 646
Dropping Index: 775
Dropping Index: 888
Dropping Index: 902
Dropping Index: 916
Dropping Index: 939
Dropping Index: 940
Dropping Index: 963
Dropping Index: 1394
Dropping Index: 1398
Dropping Index: 1399
Dropping Index: 1439
Dropping Index: 2009


In [6]:
create_license_dataset('extras/license_information/details')
client = LLMClient()

License dataset file created successfully at extras/license_information/license_dataset.csv


In [7]:
def prompt_1(comments : str, comments_extracted: bool):
    return f"""
Task: License Identification in Code Comments or Plain Text

Objective:
Extract the contiguous block (chunk) of text that is most likely to contain license-related information.

Input:
Either:
1. Comment text extracted from a code file (e.g., comments from a .cpp or .py file).
2. The full text of a file that does not typically contain comments (e.g., a NOTICE file).

Guidelines:

* IF COMMENT TEXT IS PROVIDED:
    * License information is typically contained within a single block of comment text.
    * License text often uses keywords like "copyright," "license," "permission," "terms," "distribution," "use," "modification," "attribution," etc.
    * License text might contain specific phrases like "All rights reserved," "MIT License," "Apache License 2.0," etc.

* IF PLAIN TEXT IS PROVIDED:
    * License information is likely to appear as a distinct block within the text.
    * License text is still likely to contain the same keywords and phrases as above.

Instructions:

1. Scan the input text for license-related keywords and phrases.
2. Identify contiguous blocks of text that have a higher density of these keywords.
3. Among these blocks, select all that:
    * Contain the most specific license phrases (e.g., "MIT License," "Apache License 2.0").
    * Appear most similar to typical license language (e.g., legalistic tone, reference to rights and permissions).
    * If applicable, align with any known conventions for licenses within the file type (e.g., header comments in code files).
4. If no block meets the criteria, return "No license information found."

Additional Considerations:

* Be mindful of false positives (e.g., discussions about licenses, legal notices, etc., that are not the actual license text).
* Consider the length of the block. License text can vary in length, but extremely short or overly long blocks might be less likely.
* If the input is from a code file's comments, prioritize blocks near the beginning of the file, as licenses are often placed there.

Output:

A single contiguous block of text representing the most likely license information, or "No license information found."

Output Format:
    Findings:
    [Extracted License Relevant Chunk] or [No license information found.]

{'File Comments:' if comments_extracted else 'File Text:'}
{comments}
"""

In [8]:
def prompt_2(comments : str, comments_extracted: bool):
    return f"""
Task: License Identification in Code Comments or Plain Text

Objective:
Extract the contiguous block (chunk) of text that is most likely to contain license-related information.

Input:
    Either:
    1. Comment text extracted from a code file (e.g., comments from a .cpp or .py file).
    2. The full text of a file that does not typically contain comments (e.g., a NOTICE file).

Guidelines:

* IF COMMENT TEXT IS PROVIDED:
    * License information is typically contained within a single block of comment text.
    * License text often uses keywords like "copyright," "license," "permission," "terms," "distribution," "use," "modification," "attribution," etc.
    * License text might contain specific phrases like "All rights reserved," "MIT License," "Apache License 2.0," etc.

* IF PLAIN TEXT IS PROVIDED:
    * License information is likely to appear as a distinct block within the text.
    * License text is still likely to contain the same keywords and phrases as above.

Instructions:

1. Scan the input text for license-related keywords and phrases.
2. Identify contiguous blocks of text that have a higher density of these keywords.
3. Among these blocks, select the one that:
    * Contains the most specific license phrases (e.g., "MIT License," "Apache License 2.0").
    * Appears most similar to typical license language (e.g., legalistic tone, reference to rights and permissions).
    * If applicable, aligns with any known conventions for licenses within the file type (e.g., header comments in code files).
4. If no block meets the criteria, enclose the text "No license information found." in triple backticks (```).
5. Enclose the selected block of text (or the "No license information found." message) within triple backticks (```) and return this as the output.
6. **Crucially, preserve all original spacing, indentation, and line breaks within the selected block of text.**

Additional Considerations:

* Be mindful of false positives (e.g., discussions about licenses, legal notices, etc., that are not the actual license text).
* Consider the length of the block. License text can vary in length, but extremely short or overly long blocks might be less likely.
* If the input is from a code file's comments, prioritize blocks near the beginning of the file, as licenses are often placed there.

Output:

A single contiguous block of text representing the most likely license information, or "No license information found."

{'File Comments:' if comments_extracted else 'File Text:'}
{comments}
"""

In [9]:
def prompt_3(comments: str, comments_extracted: bool):
    return f"""
Task: License Identification in Code Comments or Plain Text

Objective:
Extract the section of text that is most likely to contain license-related information. This section can be a single block of text or multiple paragraphs separated by spaces, lines, or whitespaces.

Input:
    Either:
    1. Comment text extracted from a code file (e.g., comments from a .cpp or .py file).
    2. The full text of a file that does not typically contain comments (e.g., a NOTICE file).

Guidelines:

* IF COMMENT TEXT IS PROVIDED:
    * License information may be spread across multiple comment blocks or paragraphs.
    * Focus on contiguous sections with a higher density of license keywords.
    * Look for common license phrases like "All rights reserved," "MIT License," "Apache License 2.0," etc.

* IF PLAIN TEXT IS PROVIDED:
    * License information might be present as a distinct section with potential separations.
    * Consider both keywords and phrases, as well as overall structure and context.

Instructions:

1. Scan the input text for license-related keywords and phrases.
2. Identify sections of text (single or multiple paragraphs) with a higher density of these keywords.
3. Among these sections, select the one that:
    * Contains the most specific license phrases.
    * Appears most similar to typical license language (legalistic tone, rights, permissions).
    * Aligns with conventions for licenses within the file type (if applicable).
4. If no section meets the criteria, return "No license information found."
5. Enclose the selected section (or the "No license information found." message) within triple backticks (```) and return this as the output.
6. **Crucially, preserve all original spacing, indentation, and line breaks within the selected section.**

Additional Considerations:

* Be mindful of false positives (discussions about licenses that are not the actual text).
* If the input is from code file comments, prioritize sections near the beginning.

Output:

The most likely license information as one or more text sections, or "No license information found."

{'File Comments:' if comments_extracted else 'File Text:'}
{comments}
"""


In [10]:
def prompt_4(comments : str, comments_extracted: bool):
    return f"""
Task: License Identification in Code Comments or Plain Text

Objective:
Extract the contiguous block (chunk) of text that is most likely to contain license-related information.

Input:
    Either:
    1. Comment text extracted from a code file (e.g., comments from a .cpp or .py file).
    2. The full text of a file that does not typically contain comments (e.g., a NOTICE file).

Guidelines:

* IF COMMENT TEXT IS PROVIDED:
    * License information is typically contained within a single block of comment text.
    * License text often uses keywords like "copyright," "license," "permission," "terms," "distribution," "use," "modification," "attribution," etc.
    * License text might contain specific phrases like "All rights reserved," "MIT License," "Apache License 2.0," etc.

* IF PLAIN TEXT IS PROVIDED:
    * License information is likely to appear as a distinct block within the text.
    * License text is still likely to contain the same keywords and phrases as above.

Instructions:

1. Scan the input text for license-related keywords and phrases.
2. Identify contiguous blocks of text that have a higher density of these keywords.
3. Among these blocks, select the one that:
    * Contains the most specific license phrases (e.g., "MIT License," "Apache License 2.0").
    * Appears most similar to typical license language (e.g., legalistic tone, reference to rights and permissions).
    * If applicable, aligns with any known conventions for licenses within the file type (e.g., header comments in code files).
4. If no block meets the criteria, enclose the text "No license information found." in triple backticks (```).
5. **Crucially, preserve all original spacing, indentation, and line breaks within the selected block of text.** This includes:
    * Empty lines between paragraphs
    * Leading spaces or tabs for indentation
    * Newline characters (\n) at the end of each line
6. Enclose the selected block of text (or the "No license information found." message) within triple backticks (```) and return this as the output. 

Additional Considerations:

* Be mindful of false positives (e.g., discussions about licenses, legal notices, etc., that are not the actual license text).
* Consider the length of the block. License text can vary in length, but extremely short or overly long blocks might be less likely.
* If the input is from a code file's comments, prioritize blocks near the beginning of the file, as licenses are often placed there.

Output:

A single contiguous block of text representing the most likely license information, or "No license information found."

{'File Comments:' if comments_extracted else 'File Text:'}
{comments}
"""

In [11]:
test_df.head(1)

,index,file path,licenses
0,0,NomosTestfiles/AAL/AAL.txt,AAL


In [11]:
test = extract_comments(test_df)
print(test.loc[4, 'file_comments'])

<HTML>

<HEAD>
<TITLE>Copyright and Licensing Information for ACE, TAO, CIAO, DAnCE, and CoSMIC</TITLE>

<BODY text = "#000000"
link="#000fff"
vlink="#ff0f0f"
bgcolor="#ffffff">

<HR>

<H3>Copyright and Licensing Information for ACE<sup><font
size=-2>(TM)</font></sup>, TAO<sup><font
size=-2>(TM)</font></sup>, CIAO<sup><font
size=-2>(TM)</font></sup>, DAnCE<sup><font
size=-2>(TM)</font></sup>, and
CoSMIC<sup><font
size=-2>(TM)</font></sup></H3>

<A HREF="http://www.cs.wustl.edu/~schmidt/ACE.html">ACE</a><sup><font
size=-2>(TM)</font></sup>, <A
HREF="http://www.cs.wustl.edu/~schmidt/TAO.html">TAO</A><sup><font
size=-2>(TM)</font></sup>, <A
HREF="http://www.dre.vanderbilt.edu/CIAO/">CIAO</A><sup><font
size=-2>(TM)</font></sup>, DAnCE><sup><font size=-2>(TM)</font></sup>,
and
 <A HREF="http://www.dre.vanderbilt.edu/cosmic/">CoSMIC</A><sup><font
size=-2>(TM)</font></sup> (henceforth referred to as "DOC software")
are copyrighted by <A
HREF="http://www.dre.vanderbilt.edu/~schmidt/">Douglas C

In [12]:
results_1 = client.process_dataset_license_relevant_chunks(test_df.loc[0:1], Models.LLAMA_3_8b, prompt_1,
                                                         'test-1')

/home/jimbo/Desktop/GSoC24/repo/GSoC24/helpers/llm_client.py:373: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[index, 'response'] = self._infer(model, prompt, temperature)


In [13]:
results_2 = client.process_dataset_license_relevant_chunks(test_df.loc[0:1], Models.GEMMA_2_9b, prompt_2,
                                                         'test-1')

/home/jimbo/Desktop/GSoC24/repo/GSoC24/helpers/llm_client.py:373: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[index, 'response'] = self._infer(model, prompt, temperature)


In [14]:
results_3 = client.process_dataset_license_relevant_chunks(test_df.loc[0:1], Models.LLAMA_3_1_8b, prompt_3,
                                                         'test-1')

/home/jimbo/Desktop/GSoC24/repo/GSoC24/helpers/llm_client.py:373: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[index, 'response'] = self._infer(model, prompt, temperature)


In [13]:
results_4 = client.process_dataset_license_relevant_chunks(test_df.loc[0:199], Models.TOGETHER_GEMMA_2_9b, prompt_4,
                                                         'test-1', log_every=1, retry_fails=False)

2024-08-14 11:32:20.733 | INFO     | helpers.llm_client:process_dataset_license_relevant_chunks:374 - Processing index: 0
/home/jimbo/Desktop/GSoC24/repo/GSoC24/helpers/llm_client.py:376: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[index, 'response'] = self._infer(model, prompt, temperature)
2024-08-14 11:32:26.758 | INFO     | helpers.llm_client:process_dataset_license_relevant_chunks:374 - Processing index: 1
2024-08-14 11:32:32.602 | INFO     | helpers.llm_client:process_dataset_license_relevant_chunks:374 - Processing index: 2
2024-08-14 11:32:34.177 | INFO     | helpers.llm_client:process_dataset_license_relevant_chunks:374 - Processing index: 3
2024-08-14 11:32:38.289 | INFO     | helpers.llm_client:process_dataset_license_relevant

In [14]:
print(results_4.loc[0, 'response'])

```
ATTRIBUTION ASSURANCE LICENSE (adapted from the original BSD license)
Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the conditions below are met.
These conditions require a modest attribution to <AUTHOR> (the
"Author"), who hopes that its promotional value may help justify the
thousands of dollars in otherwise billable time invested in writing
this and other freely available, open-source software.

1. Redistributions of source code, in whole or part and with or without
modification (the "Code"), must prominently display this GPG-signed
text in verifiable form.
2. Redistributions of the Code in binary form must be accompanied by
this GPG-signed text in any documentation and, each time the resulting
executable program or a program dependent thereon is launched, a
prominent display (e.g., splash screen or banner text) of the Author's
attribution information, which includes:
(a) Name ("AUTHOR"),
(b) Professional identificat

In [38]:
print(results_4.loc[4, 'file_comments'])

<HTML>

<HEAD>
<TITLE>Copyright and Licensing Information for ACE, TAO, CIAO, DAnCE, and CoSMIC</TITLE>

<BODY text = "#000000"
link="#000fff"
vlink="#ff0f0f"
bgcolor="#ffffff">

<HR>

<H3>Copyright and Licensing Information for ACE<sup><font
size=-2>(TM)</font></sup>, TAO<sup><font
size=-2>(TM)</font></sup>, CIAO<sup><font
size=-2>(TM)</font></sup>, DAnCE<sup><font
size=-2>(TM)</font></sup>, and
CoSMIC<sup><font
size=-2>(TM)</font></sup></H3>

<A HREF="http://www.cs.wustl.edu/~schmidt/ACE.html">ACE</a><sup><font
size=-2>(TM)</font></sup>, <A
HREF="http://www.cs.wustl.edu/~schmidt/TAO.html">TAO</A><sup><font
size=-2>(TM)</font></sup>, <A
HREF="http://www.dre.vanderbilt.edu/CIAO/">CIAO</A><sup><font
size=-2>(TM)</font></sup>, DAnCE><sup><font size=-2>(TM)</font></sup>,
and
 <A HREF="http://www.dre.vanderbilt.edu/cosmic/">CoSMIC</A><sup><font
size=-2>(TM)</font></sup> (henceforth referred to as "DOC software")
are copyrighted by <A
HREF="http://www.dre.vanderbilt.edu/~schmidt/">Douglas C

In [32]:
results_4_ = results_4[results_4['response'].notna()]

In [34]:
import difflib

def is_similar_substring(substring, mainstring, threshold=0.9):
    matcher = difflib.SequenceMatcher(None, substring, mainstring)
    matching_blocks = matcher.get_matching_blocks()
    matching_text_length = sum(block.size for block in matching_blocks)
    similarity_ratio = matching_text_length / len(substring)  
    return similarity_ratio >= threshold


for index, row in results_4_.iterrows():
    file_comments = row['file_comments']
    response = row['response']
    code_blocks = re.findall(r"```.*```", response, re.DOTALL)  
    for block in code_blocks:
        if not is_similar_substring(block, file_comments, threshold=0.9):
            results_4_.loc[index, 'license_relevant_chunk_found'] = False
            print(f"No similar code block found in file_comments for index {index}:")
        else:
            results_4_.loc[index, 'license_relevant_chunk_found'] = True
results_4_['license_relevant_chunk_found'] = results_4_['license_relevant_chunk_found'].apply(lambda x: x if x in [True, False] else True)
print('Extraction Accuracy: ', accuracy_score([True] * len(results_4_), results_4_['license_relevant_chunk_found'].to_list()))

No similar code block found in file_comments for index 23:
No similar code block found in file_comments for index 25:
No similar code block found in file_comments for index 29:
No similar code block found in file_comments for index 31:
No similar code block found in file_comments for index 32:
No similar code block found in file_comments for index 55:
No similar code block found in file_comments for index 56:
No similar code block found in file_comments for index 65:
No similar code block found in file_comments for index 73:
No similar code block found in file_comments for index 106:
No similar code block found in file_comments for index 120:
No similar code block found in file_comments for index 122:
No similar code block found in file_comments for index 124:
No similar code block found in file_comments for index 139:
No similar code block found in file_comments for index 141:
No similar code block found in file_comments for index 165:
No similar code block found in file_comments for 

/tmp/ipykernel_46958/3462747271.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_4_['license_relevant_chunk_found'] = results_4_['license_relevant_chunk_found'].apply(lambda x: x if x in [True, False] else True)


In [14]:
results_df = pd.read_csv('results/test-1.csv') # in spreadsheet - 2 (2 -> 0, etc.)

In [21]:
print(results_df.loc[27, 'response'])

nan


In [45]:
print(results_4_.loc[19, 'response'])

```
Academic Free License ("AFL") v. 3.0

This Academic Free License (the "License") applies to any original work
of authorship (the "Original Work") whose owner (the "Licensor") has
placed the following licensing notice adjacent to the copyright notice
for the Original Work:

Licensed under the Academic Free License version 3.0

1) Grant of Copyright License. Licensor grants You a worldwide,
royalty-free, non-exclusive, sublicensable license, for the duration of
the copyright, to do the following:

	a) to reproduce the Original Work in copies, either alone or as
	   part of a collective work;

	b) to translate, adapt, alter, transform, modify, or arrange the
	   Original Work, thereby creating derivative works ("Derivative
	   Works") based upon the Original Work;

	c) to distribute or communicate copies of the Original Work and
	   Derivative Works to the public, UNDER ANY LICENSE OF YOUR
	   CHOICE THAT DOES NOT CONTRADICT THE TERMS AND CONDITIONS,
	   INCLUDING LICENSOR'S RESERVED 

In [46]:
print(results_4_.loc[19, 'file path'], results_4_.loc[19, 'licenses'])

NomosTestfiles/AFL/LICENSE AFL-3.0


In [47]:
print(results_4.loc[19, 'file_comments'])

Most of MASON is licensed under the Academic Free License, version 3.0,
with the exception of the following files:

	PngEncoder.java			Artistic License
	MovieEncoder.java		[Partial] Sun Open Source License
	WireFrameBoxPortrayal3D.java	[Partial] Sun Open Source License
	ToolTipBehavior.java		[Partial] Sun Open Source License
	SelectionBehavior.java		[Partial] Sun Open Source License
	GullCG.java			Sun Open Source License
	MersenneTwisterFast.java	BSD License

For all remaining files, the Academic Free License is specified below.


Academic Free License ("AFL") v. 3.0

This Academic Free License (the "License") applies to any original work
of authorship (the "Original Work") whose owner (the "Licensor") has
placed the following licensing notice adjacent to the copyright notice
for the Original Work:

Licensed under the Academic Free License version 3.0

1) Grant of Copyright License. Licensor grants You a worldwide,
royalty-free, non-exclusive, sublicensable license, for the duration of

In [4]:
meh = pd.read_csv('processedLicenses.csv')

In [5]:
meh.columns

Index(['shortname', 'fullname', 'text', 'license_header', 'url', 'deprecated',
       'osi_approved', 'isException', 'processed_fullname', 'processed_header',
       'processed_text'],
      dtype='object')

In [6]:
meh[meh['shortname'] == 'LAL-1.2']

,shortname,fullname,text,license_header,url,deprecated,osi_approved,isException,processed_fullname,processed_header,processed_text
402,LAL-1.2,Licence Art Libre 1.2,Licence Art Libre\n[ Copyleft Attitude ]\n\nVe...,NaN,http://artlibre.org/licence/lal/licence-art-li...,False,False,False,licence art libre 1 2,NaN,licence art libre copyleft attitude version 1 ...


In [7]:
print(meh.loc[402, 'text'] )

Licence Art Libre
[ Copyleft Attitude ]

Version 1.2

Préambule :

Avec cette Licence Art Libre, l’autorisation est donnée de copier, de diffuser et de transformer librement les oeuvres dans le respect des droits de l’auteur.

Loin d’ignorer les droits de l’auteur, cette licence les reconnaît et les protège. Elle en reformule le principe en permettant au public de faire un usage créatif des oeuvres d’art.
Alors que l’usage fait du droit de la propriété littéraire et artistique conduit à restreindre l’accès du public à l’oeuvre, la Licence Art Libre a pour but de le favoriser.
L’intention est d’ouvrir l’accès et d’autoriser l’utilisation des ressources d’une oeuvre par le plus grand nombre. En avoir jouissance pour en multiplier les réjouissances, créer de nouvelles conditions de création pour amplifier les possibilités de création. Dans le respect des auteurs avec la reconnaissance et la défense de leur droit moral.

En effet, avec la venue du numérique, l’invention de l’internet et de

In [25]:
print(meh.loc[45, 'license_header'] )

Copyright [yyyy] [name of copyright owner]

Licensed under the Apache License, Version 2.0 (the "License");

you may not use this file except in compliance with the License.

You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software

distributed under the License is distributed on an "AS IS" BASIS,

WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.

See the License for the specific language governing permissions and

limitations under the License.




In [31]:
meh1 = pd.read_csv('extras/license_information/license_dataset.csv')
meh1.loc[0, 'License Text']

'\nSPDX-License-Identifier: CNRI-Jython\n\nLicense Name: CNRI Jython License\n\n\n1. This LICENSE AGREEMENT is between the Corporation for National Research Initiatives, having an office at 1895 Preston White Drive, Reston, VA 20191 ("CNRI"), and the Individual or Organization ("Licensee") accessing and using JPython version 1.1.x in source or binary form and its associated documentation as provided herein ("Software").\n\n2.  Subject to the terms and conditions of this License Agreement, CNRI hereby grants Licensee a non-exclusive, non-transferable, royalty-free, world-wide license to reproduce, analyze, test, perform and/or display publicly, prepare derivative works, distribute, and otherwise use the Software alone or in any derivative version, provided, however, that CNRI\'s License Agreement and CNRI\'s notice of copyright, i.e., “Copyright (c) 1996-1999 Corporation for National Research Initiatives; All Rights Reserved” are both retained in the Software, alone or in any derivative

In [32]:
license_index_map = {}
all_license_texts = []
for license_index, license_text in enumerate(meh1['License Text']):
    for line in license_text.split('\n'):
        all_license_texts.append(line)
        license_index_map[len(all_license_texts) - 1] = license_index

In [34]:
print(all_license_texts[0:10])

['', 'SPDX-License-Identifier: CNRI-Jython', '', 'License Name: CNRI Jython License', '', '', '1. This LICENSE AGREEMENT is between the Corporation for National Research Initiatives, having an office at 1895 Preston White Drive, Reston, VA 20191 ("CNRI"), and the Individual or Organization ("Licensee") accessing and using JPython version 1.1.x in source or binary form and its associated documentation as provided herein ("Software").', '', "2.  Subject to the terms and conditions of this License Agreement, CNRI hereby grants Licensee a non-exclusive, non-transferable, royalty-free, world-wide license to reproduce, analyze, test, perform and/or display publicly, prepare derivative works, distribute, and otherwise use the Software alone or in any derivative version, provided, however, that CNRI's License Agreement and CNRI's notice of copyright, i.e., “Copyright (c) 1996-1999 Corporation for National Research Initiatives; All Rights Reserved” are both retained in the Software, alone or in